# BeeGuard — Zero-Shot Auto-Labeling (Grounding DINO)

**Read this first:** this generates *draft* bounding boxes using text prompts, not final labels.
You will still open the output in CVAT and correct it. Expect:
- `Queen_Bee`, `Capped_brood`: reasonable hit rate, worth reviewing and keeping most
- `Larvae`, `Drone_bee_Capped_brood`: mixed — expect to delete false positives and add missed ones
- `Eggs`: likely poor — tiny objects, review carefully, don't trust counts
- `Healthy_brood_pattern`: **not attempted here** — this is a pattern across many cells, not a single detectable object. You'll need to draw these boxes manually in CVAT (e.g. box around a comb region with solid, even capping and no gaps).

**Runtime:** `Runtime -> Change runtime type -> T4 GPU`

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies

In [ ]:
!pip install -q transformers accelerate pillow

## 3. Upload the frames you want auto-labeled

On your local machine: zip up whichever unlabeled frames from `D:\BeeGuard_Videos\Frames\` you want
labeled (e.g. a batch of ~200-300 you haven't annotated yet), upload the zip to
`MyDrive/BeeGuard/unlabeled_frames.zip`.

In [ ]:
FRAMES_ZIP = "/content/drive/MyDrive/BeeGuard/unlabeled_frames.zip"

!mkdir -p /content/unlabeled_frames
!unzip -q "$FRAMES_ZIP" -d /content/unlabeled_frames
!find /content/unlabeled_frames -maxdepth 2 -type f | head -5
!find /content/unlabeled_frames -type f | wc -l

## 4. Load Grounding DINO

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_id = "IDEA-Research/grounding-dino-tiny"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

## 5. Define class prompts

Order and IDs match your real `obj.names` exactly. `Healthy_brood_pattern` is deliberately
excluded from auto-detection (see note above) but kept in the ID mapping so the numbers line
up with your existing 50 labeled frames.

Grounding DINO expects prompts as lowercase phrases separated by periods.

In [ ]:
# class_id : (prompt phrase, confidence threshold)
# Thresholds are set per-class: lower for classes that are hard to catch at all
# (accept more false positives you'll delete), higher for classes that are easy to
# confuse with the background (avoid drowning yourself in junk boxes to clean up).
CLASS_PROMPTS = {
    0: ("capped brood cell.", 0.30),           # Capped_brood
    1: ("drone brood cell with domed cap.", 0.25),  # Drone_bee_Capped_brood
    3: ("bee egg.", 0.20),                     # Eggs — expect weak results
    4: ("bee larva.", 0.25),                   # Larvae
    5: ("queen bee.", 0.30),                   # Queen_Bee
    # 2: Healthy_brood_pattern intentionally skipped
}

CLASS_NAMES = {
    0: "Capped_brood",
    1: "Drone_bee_Capped_brood",
    2: "Healthy_brood_pattern",
    3: "Eggs",
    4: "Larvae",
    5: "Queen_Bee",
}

## 6. Run detection and write YOLO-format labels

In [ ]:
import os
from pathlib import Path
from PIL import Image
from tqdm import tqdm

FRAMES_DIR = Path("/content/unlabeled_frames")
OUT_LABELS_DIR = Path("/content/auto_labels")
OUT_LABELS_DIR.mkdir(exist_ok=True)

image_paths = sorted([p for p in FRAMES_DIR.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")])
print(f"Found {len(image_paths)} images to auto-label.")

detection_counts = {name: 0 for name in CLASS_NAMES.values()}

for img_path in tqdm(image_paths):
    image = Image.open(img_path).convert("RGB")
    img_w, img_h = image.size
    lines = []

    for class_id, (prompt, threshold) in CLASS_PROMPTS.items():
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)

        results = processor.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            box_threshold=threshold,
            text_threshold=threshold,
            target_sizes=[(img_h, img_w)],
        )[0]

        for box in results["boxes"]:
            x0, y0, x1, y1 = box.tolist()
            center_x = (x0 + x1) / 2.0 / img_w
            center_y = (y0 + y1) / 2.0 / img_h
            width = (x1 - x0) / img_w
            height = (y1 - y0) / img_h
            lines.append(f"{class_id} {center_x:.6f} {center_y:.6f} {width:.6f} {height:.6f}")
            detection_counts[CLASS_NAMES[class_id]] += 1

    label_path = OUT_LABELS_DIR / (img_path.stem + ".txt")
    with open(label_path, "w") as f:
        if lines:
            f.write("\n".join(lines) + "\n")

print("\nDraft detections per class (before you review/correct):")
for name, count in detection_counts.items():
    print(f"  {name}: {count}")

## 7. Zip and save results to Drive

In [ ]:
import shutil

shutil.make_archive("/content/auto_labels", "zip", "/content/auto_labels")

!mkdir -p /content/drive/MyDrive/BeeGuard
!cp /content/auto_labels.zip /content/drive/MyDrive/BeeGuard/auto_labels.zip
print("Saved to Drive: MyDrive/BeeGuard/auto_labels.zip")

## 8. (Optional) Download directly instead of using Drive

In [ ]:
from google.colab import files
files.download("/content/auto_labels.zip")

## Next step: review in CVAT

1. In CVAT, create a task using the same images you auto-labeled here.
2. Import `auto_labels.zip` as a **YOLO 1.1** annotation format upload — this pre-fills
   boxes as a starting point.
3. Go through every frame. For `Eggs` especially, assume the model missed most real eggs
   and may have flagged non-egg specks — you'll be adding boxes more than deleting them.
4. Manually draw `Healthy_brood_pattern` boxes yourself — there are no auto-generated
   boxes for this class.
5. Export corrected annotations from CVAT, merge with your existing 50 labeled frames,
   then re-run `split_dataset.py` with the combined set before retraining.